# Unidad: Desarrollo de Aplicaciones de IA - Benchmarking y Optimización
Este cuaderno explora el rendimiento de modelos de IA utilizando aceleración por hardware (**CUDA**). Analizaremos el fenómeno del *warm-up*, el uso de *pipelines* de NLP y el cálculo matricial de alto rendimiento.

**Objetivo:** Comprender la interacción entre el software de IA y el hardware de aceleración.

## 1. El fenómeno del "Warm-up" en GPUs
Al trabajar con **PyTorch** y **CUDA**, la primera inferencia suele ser significativamente más lenta. Esto se debe a la inicialización del contexto de la GPU, la carga de kernels de C++/CUDA en la VRAM y la optimización de cuDNN.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 740.1 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 734.0 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 199.6/663.9 MB 449.9 kB/s eta 0:17:13

In [ ]:
'''
En la lista de arquitecturas soportadas por tu binario instalado (sm_70, sm_75, etc.),
falta la arquitectura sm_61, que es la que corresponde a tu MX350 (Pascal)

¿CUDA disponible?: True
Nombre: NVIDIA GeForce MX350
Capacidad de Cómputo: 6.1
Arquitecturas soportadas por este binario: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
'''
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [20]:
import torch

def diagnose_cuda():
    print(f"¿CUDA disponible?: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        print(f"Nombre: {prop.name}")
        print(f"Capacidad de Cómputo: {prop.major}.{prop.minor}") # Debería salir 6.1
        print(f"Arquitecturas soportadas por este binario: {torch.cuda.get_arch_list()}")
        
diagnose_cuda()

¿CUDA disponible?: True
Nombre: NVIDIA GeForce MX350
Capacidad de Cómputo: 6.1
Arquitecturas soportadas por este binario: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [ ]:
import torch

def diagnose_cuda():
    print(f"¿CUDA disponible?: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        print(f"Nombre: {prop.name}")
        print(f"Capacidad de Cómputo: {prop.major}.{prop.minor}") # Debería salir 6.1
        print(f"Arquitecturas soportadas por este binario: {torch.cuda.get_arch_list()}")
        
diagnose_cuda()

In [17]:
import torchvision.models as models
import time
from torchvision.utils import save_image

device = torch.device("cuda")
model = models.resnet50(weights='DEFAULT').to(device)
model.eval()

# impasss:  el tensor representa a 1 imagen, de 3 canalres de 224 por 224
# randn es un número aleatorio siguiendo la Normal(0,1)
imagen_no_normalida = torch.randn(1, 3, 224, 224)

# hay que normalizar para imprimirla 
# las imágenes tienen en los pixeles de cada color(RGB) entre 0 y 1 
# y N(0,1) va de -infinito a + infinito.
# Normalizar a [0,1]
img = (imagen_no_normalida - imagen_no_normalida.min()) / (imagen_no_normalida.max() - imagen_no_normalida.min())

# Guardar
save_image(img, "imagen.png")
#Fin del impass

dummy_input = imagen_no_normalida.to(device)

# Primera inferencia (Warm-up)
torch.cuda.synchronize()
start = time.time()
with torch.no_grad():
    _ = model(dummy_input)
torch.cuda.synchronize()
print(f"Tiempo Warm-up: {time.time() - start:.4f}s")

# Bucle de inferencia
torch.cuda.synchronize()
start = time.time()
for _ in range(50):
    with torch.no_grad():
        _ = model(dummy_input)
torch.cuda.synchronize()
print(f"Tiempo medio (50 iters): {(time.time() - start)/50:.4f}s")

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 2. NLP a gran escala con Hugging Face
Utilizaremos la librería `transformers` para delegar el cómputo de los *Attention Layers* a los núcleos CUDA. Procesaremos el dataset de críticas de cine de **IMDb**.

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import time

# Carga de datos (200 registros para testing)
dataset = load_dataset("imdb", split="test[:200]")

# Inicialización del pipeline en GPU (device=0)
classifier = pipeline("sentiment-analysis", 
                      model="distilbert-base-uncased-finetuned-sst-2-english", 
                      device=0)

# Ejecución por lotes para maximizar el throughput
start = time.time()
results = classifier(dataset["text"], batch_size=16)
end = time.time()

print(f"Procesadas 200 reviews en: {end - start:.2f} segundos")
print(f"Ejemplo de salida: {results[0]}")

## 3. CuPy: Aceleración Matricial Transparente
Comparativa de rendimiento entre CPU (NumPy) y GPU (CuPy) para operaciones de álgebra lineal masiva.

In [ ]:
import numpy as np
import cupy as cp
import time

size = 8000

# Benchmark NumPy (CPU)
start_cpu = time.time()
a_cpu = np.random.random((size, size)).astype(np.float32)
b_cpu = np.random.random((size, size)).astype(np.float32)
res_cpu = np.dot(a_cpu, b_cpu)
t_cpu = time.time() - start_cpu

# Benchmark CuPy (GPU)
start_gpu = time.time()
a_gpu = cp.random.random((size, size)).astype(cp.float32)
b_gpu = cp.random.random((size, size)).astype(cp.float32)
res_gpu = cp.dot(a_gpu, b_gpu)
cp.cuda.Device(0).synchronize()
t_gpu = time.time() - start_gpu

print(f"Tiempo CPU (NumPy): {t_cpu:.4f}s")
print(f"Tiempo GPU (CuPy): {t_gpu:.4f}s")
print(f"Aceleración (Speed-up): {t_cpu/t_gpu:.2f}x")